In [ ]:
import wget

url = "https://github.com/GrandmaCan/ML/raw/main/Classification/one_piece_mini.zip"
wget.download(url)

In [ ]:
import zipfile

with zipfile.ZipFile("one_piece_mini.zip", "r") as zip_ref:
    zip_ref.extractall("one_piece_mini")

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [ ]:
from torch.utils.data import Dataset
from pathlib import Path
from PIL import Image
from torchvision import transforms

class ImageDataset(Dataset):
    def __init__(self, root, train, transform=None):
        if train:
            image_root = Path(root) / "train"
        else:
            image_root = Path(root) / "test"
        
        with open(Path(root) / "classnames.txt", "r") as f:
            lines = f.readlines()
            self.classes = [line.strip() for line in lines]
        self.paths = [i for i in image_root.rglob("*") if i.is_file()]
        self.transform = transform

    def __getitem__(self, index):
        image = Image.open(self.paths[index]).convert('RGB')
        class_name = self.paths[index].parent.name
        class_index = self.classes.index(class_name)

        if self.transform:
            return self.transform(image), class_index
        else:
            return image, class_index
    def __len__(self):
        return len(self.paths)

In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize((64, 64)), 
    transforms.TrivialAugmentWide(), 
    transforms.ToTensor()
])

test_transforms = transforms.Compose([
    transforms.Resize((64, 64)), 
    transforms.ToTensor()
])

In [ ]:
train_data = ImageDataset(
    'one_piece_mini', 
    train=True, 
    transform=train_transforms
)

test_data = ImageDataset(
    'one_piece_mini', 
    train=False, 
    transform=test_transforms
)

x, y = train_data[40]
x.shape, y

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(x.permute(1, 2, 0))

In [ ]:
len(train_data), len(test_data)

In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 8

train_dataloader = DataLoader(train_data, 
                                batch_size=BATCH_SIZE,
                                shuffle=True)
test_dataloader = DataLoader(test_data, 
                                batch_size=BATCH_SIZE,
                                shuffle=False)
len(train_dataloader), len(test_dataloader)

In [ ]:
x_first_batch, y_first_batch = next(iter(train_dataloader))
x_first_batch.shape, y_first_batch.shape

In [ ]:
from torch import nn

class ImageClassificationModel(nn.Module):
    def __init__(self, input_shape, output_shape):
        super().__init__()
        self.conv_block_1 = nn.Sequential(
            nn.Conv2d(in_channels=input_shape, 
                        out_channels=8, 
                        kernel_size=(3, 3), 
                        stride=1, 
                        padding=1
            ),
            nn.ReLU(), 
            nn.Conv2d(in_channels=8, 
                        out_channels=8, 
                        kernel_size=(3, 3), 
                        stride=1,
                        padding=1
            ),
            nn.ReLU(), 
            nn.MaxPool2d(kernel_size=(2, 2), 
                            stride=2, 
                            padding=0
            )
        )

        self.conv_block_2 = nn.Sequential(
            nn.Conv2d(in_channels=8, 
                        out_channels=16, 
                        kernel_size=(3, 3), 
                        stride=1, 
                        padding=1
            ),
            nn.ReLU(), 
            nn.Conv2d(in_channels=16, 
                        out_channels=16, 
                        kernel_size=(3, 3), 
                        stride=1,
                        padding=1
            ),
            nn.ReLU(), 
            nn.MaxPool2d(kernel_size=(2, 2), 
                            stride=2, 
                            padding=0
            )
        )

        self.classifier = nn.Sequential(
            nn.Flatten(start_dim=1, end_dim=-1), # 不能從第零個維度開始攤平，因為第零個維度是batch的數量。
            nn.Linear(in_features=16*16*16, out_features=output_shape)
        )
    def forward(self, x):
        x = self.conv_block_1(x)
        x = self.conv_block_2(x)
        x = self.classifier(x)
        return x

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
model = ImageClassificationModel(3, len(train_data.classes))
model.to(device)
model(x_first_batch.to(device))

In [ ]:
def accuracy_fn(y_pred, y_true):
    correct_num = (y_pred==y_true).sum()
    acc = correct_num / len(y_pred) * 100

    return acc

In [ ]:
cost_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
def train_step(dataloader, model, cost_fn, optimizer, device):
    train_cost = 0
    train_acc = 0

    model.train()

    for batch, (x, y) in enumerate(dataloader):# 每個batch分別訓練
        x = x.to(device)
        y = y.to(device)

        y_pred = model(x)
        cost = cost_fn(y_pred, y)

        train_cost += cost.item()
        train_acc +=accuracy_fn(y_pred.argmax(dim=1), y)

        optimizer.zero_grad()
        cost.backward()
        optimizer.step()

    train_cost /= len(dataloader)#train_cost = train_cost / len(train_dataloader)
    train_acc /= len(dataloader)#做平均
    print(f"\nTrain Cost: {train_cost:.4f}, Train Acc: {train_acc:.2f}")


def test_step(dataloader, model, cost_fn, device):
    test_cost = 0
    test_acc = 0
    model.eval()
    with torch.inference_mode():
        for x, y in dataloader:
            x = x.to(device)
            y = y.to(device)

            test_pred = model(x)

            test_cost += cost_fn(test_pred, y)
            test_acc += accuracy_fn(test_pred.argmax(dim=1), y)

        test_cost /= len(dataloader)
        test_acc /= len(dataloader)#做平均

    print(f"Test Cost: {test_cost:.4f}, Test Acc: {test_acc:.2f}\n")

In [ ]:
epochs = 1

for epoch in range(epochs):
    print(f'Epoch: {epoch}\n-------------')

    train_step(train_dataloader, model, cost_fn, optimizer, device)
    test_step(test_dataloader, model, cost_fn, device)

In [ ]:
image = Image.open('nami.jpg').convert('RGB')
image = test_transforms(image)
image = image.reshape(-1, 3, 64, 64)

model.eval()
with torch.inference_mode():
    y_pred = model(image.to(device))
y_pred = torch.softmax(y_pred, dim=1)
class_idx = y_pred.argmax(dim=1)
test_data.classes[class_idx]

改使用遷移學習

In [ ]:
import requests

url = "https://firebasestorage.googleapis.com/v0/b/grandmacan-2dae4.appspot.com/o/ML_data%2Fone_piece_full.zip?alt=media&token=937656fd-f5c1-44f5-b174-1e2d590b8ef3"

with open("one_piece_full.zip", "wb") as f:
  req = requests.get(url)
  f.write(req.content)

In [ ]:
import zipfile

with zipfile.ZipFile("one_piece_full.zip", "r") as zip_file:
    zip_file.extractall("one_piece_full")

In [ ]:
train_dataset = ImageDataset(root="one_piece_full", 
                train=True, 
                transform=train_transforms
)

test_dataset = ImageDataset(root="one_piece_full", 
                train=False, 
                transform=test_transforms
)

In [ ]:
len(train_dataset), len(test_dataset)

In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 16

train_dataloader = DataLoader(dataset=train_dataset,
                batch_size=BATCH_SIZE,
                shuffle=True
)

test_dataloader = DataLoader(dataset=test_dataset,
                batch_size=BATCH_SIZE,
                shuffle=False
)

In [ ]:
len(train_dataloader), len(test_dataloader)

In [ ]:
model_2 = ImageClassificationModel(3, len(train_dataset.classes))
model_2.to(device)

In [ ]:
cost_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model_2.parameters(), lr=0.001)

In [ ]:
from tqdm.auto import tqdm

epochs = 10

for epoch in tqdm(range(epochs)):
    print(f"Epoch: {epoch}\n-------")

    train_step(train_dataloader, model_2, cost_fn, optimizer, accuracy_fn, device)

    test_step(test_dataloader, model_2, cost_fn, accuracy_fn, device)